In [27]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
from package.llms.ollama import OllamaLLM, OpenAIOutputMessage, LlamaOutputMessage
from package.llms.bedrock import BedrockNova
# import logging
# from package.utils import setup_logger
from package.prompt_hub import PromptHub

# setup_logger(logging.DEBUG)

# openai_llm = OllamaLLM(model_id="gpt-oss:20b", OutputMessage=OpenAIOutputMessage)
# llama_llm = OllamaLLM(model_id="llama3.2", OutputMessage=LlamaOutputMessage)
nova_llm = BedrockNova(model_id="us.amazon.nova-micro-v1:0")

In [108]:
from package.intent_hub.query import query_general
from package.intent_hub.nextaction import nextaction

from package.agents.intent_agent import MultiIntentClassifier

model = MultiIntentClassifier()
model.add_examples(intent="query", examples=query_general)
model.add_examples(intent="nextaction", examples=nextaction)

In [111]:
# text = "อยากทราบยอดขายเดือนนี้"
# text = "ควยครับ"
# text = "หิวข้าวอยากกินเต๊๋วเรือ"
# text = "ยอดขายเดือนนี้เป็นอย่างไร อะไรโตสูงสุด"
# text = "ประกันอะไรยอดขายตกในปีนี้"
# text = "จากข้อมูลนี้ควรเอาไปใช้ยังไงต่อ"
# text = "เอาไปทำอะไรต่อดี"
# text = "โย่วๆยอดขายเดือนนี้เป็นไงบ้างวะไอสาด"
# text = "โย่วๆประกันเหี้ยไรคนสนใจเยอะวะไอสาด"
# text = "ประกันไรคนสนใจเยอะ"
# text = "ประกันไรคนซื้อเยอะวะควยเอ้ย"
# text = "สินค้าไรคนซื้อเยอะวันนี้วะควยเอ้ย"
model.predict(text, threshold=0.7)

('unknown_intent',
 {'nextaction': 0.454879789352417, 'query': 0.43011768579483034})

# plot visualize

In [ ]:
import plotly.io as pio
pio.renderers.default = "browser"

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import umap
import plotly.express as px

import plotly.io as pio
pio.renderers.default = "browser"

X = np.stack(model.vectors)
y = model.labels
texts = query_general+nextaction

reducer = umap.UMAP(

    
    n_components=2,
    # n_neighbors=15,
    # min_dist=0.5,
    metric="cosine",
    random_state=42
)

X_2d = reducer.fit_transform(X)

# สร้าง DataFrame เพื่อคำนวณ centroid ง่าย ๆ
df = pd.DataFrame({
    "x": X_2d[:, 0],
    "y": X_2d[:, 1],
    "label": y,
    "text": texts
})

# คำนวณ centroid ต่อ label
centroids = (
    df.groupby("label")[["x", "y"]]
    .mean()
    .reset_index()
)

# scatter จุดปกติ
fig = px.scatter(
    df,
    x="x",
    y="y",
    color="label",
    hover_data={"text": True},
    labels={
        "x": "UMAP-1",
        "y": "UMAP-2",
        "label": "label"
    }
)

# เพิ่ม centroid ลงไป
fig.add_trace(
    go.Scatter(
        x=centroids["x"],
        y=centroids["y"],
        mode="markers+text",
        text=centroids["label"],
        textposition="top center",
        marker=dict(
            symbol="x",
            size=16,
            line=dict(width=2)
        ),
        name="centroid"
    )
)

fig.show()


d:\Git\leonidas-arthena\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



# optimize

In [119]:
import optuna
import numpy as np

# scores = list of cosine similarity of validation samples for this intent
# y_true = list 0/1 ว่าทาย intent ถูกไหม
def f1_given_threshold(scores, y_true, th):
    y_pred = [1 if s >= th else 0 for s in scores]
    tp = sum((p==1 and t==1) for p,t in zip(y_pred,y_true))
    fp = sum((p==1 and t==0) for p,t in zip(y_pred,y_true))
    fn = sum((p==0 and t==1) for p,t in zip(y_pred,y_true))
    if tp == 0:
        return 0
    precision = tp/(tp+fp)
    recall = tp/(tp+fn)
    f = 1
    return f*precision*recall/((f**2)*precision+recall)

def optimize_threshold(scores, y_true):
    def objective(trial):
        th = trial.suggest_float("threshold", 0.0, 1.0)
        return -f1_given_threshold(scores, y_true, th)

    study = optuna.create_study()
    study.optimize(objective, n_trials=1000)
    return study.best_params["threshold"], -study.best_value

## query

In [117]:
from package.intent_hub.query import query_general
from package.intent_hub.nextaction import nextaction

from package.agents.intent_agent import MultiIntentClassifier

model = MultiIntentClassifier()
model.add_examples(intent="query", examples=query_general)
# model.add_examples(intent="nextaction", examples=nextaction)

In [120]:
from package.intent_hub.query_test import query_test
from package.intent_hub.nextaction_test import nextaction_test
from package.intent_hub.noise import noise

scores_intent = []  # cosine ของ intent นี้
y_true_intent = []  # 1/0

for text in query_test:
    _ = model.predict(text, threshold=0.7)
    # if _[0]=='query': 
    #     y_true_intent.append(1) 
    # else: 
    #     y_true_intent.append(0)
    y_true_intent.append(1)
    scores_intent.append(_[1]['query'])

for text in nextaction+nextaction_test+noise:
    _ = model.predict(text, threshold=0.7)
    # if _[0]=='query': 
    #     y_true_intent.append(1) 
    # else: 
    #     y_true_intent.append(0)
    y_true_intent.append(0)
    scores_intent.append(_[1]['query'])

best_th, best_f1 = optimize_threshold(scores_intent, y_true_intent)
print(best_th, best_f1)

[I 2025-12-14 20:39:25,253] A new study created in memory with name: no-name-598c59d1-52a4-4316-bd47-80aeaff14ca0
[I 2025-12-14 20:39:25,255] Trial 0 finished with value: -0.22346368715083798 and parameters: {'threshold': 0.17225592756559926}. Best is trial 0 with value: -0.22346368715083798.
[I 2025-12-14 20:39:25,255] Trial 1 finished with value: -0.22346368715083798 and parameters: {'threshold': 0.22803983999057098}. Best is trial 0 with value: -0.22346368715083798.
[I 2025-12-14 20:39:25,256] Trial 2 finished with value: -0.22346368715083798 and parameters: {'threshold': 0.13341835000755875}. Best is trial 0 with value: -0.22346368715083798.
[I 2025-12-14 20:39:25,256] Trial 3 finished with value: -0.22346368715083798 and parameters: {'threshold': 0.20517151795159805}. Best is trial 0 with value: -0.22346368715083798.
[I 2025-12-14 20:39:25,257] Trial 4 finished with value: -0.312 and parameters: {'threshold': 0.6483528253428694}. Best is trial 4 with value: -0.312.
[I 2025-12-14 2

0.7079720464652063 0.35238095238095235


## next action

In [121]:
from package.intent_hub.query import query_general
from package.intent_hub.nextaction import nextaction

from package.agents.intent_agent import MultiIntentClassifier

model = MultiIntentClassifier()
# model.add_examples(intent="query", examples=query_general)
model.add_examples(intent="nextaction", examples=nextaction)

In [122]:
from package.intent_hub.nextaction_test import nextaction_test
from package.intent_hub.query_test import query_test
from package.intent_hub.noise import noise

scores_intent = []  # cosine ของ intent นี้
y_true_intent = []  # 1/0

for text in nextaction_test:
    _ = model.predict(text, threshold=0.7)
    # if _[0]=='query': 
    #     y_true_intent.append(1) 
    # else: 
    #     y_true_intent.append(0)
    y_true_intent.append(1)
    scores_intent.append(_[1]['nextaction'])

for text in query_general+query_test+noise:
    _ = model.predict(text, threshold=0.7)
    # if _[0]=='query': 
    #     y_true_intent.append(1) 
    # else: 
    #     y_true_intent.append(0)
    y_true_intent.append(0)
    scores_intent.append(_[1]['nextaction'])

best_th, best_f1 = optimize_threshold(scores_intent, y_true_intent)
print(best_th, best_f1)

[I 2025-12-14 20:39:47,030] A new study created in memory with name: no-name-84c2e28f-d9be-4cf5-949b-9563dccf6718
[I 2025-12-14 20:39:47,031] Trial 0 finished with value: -0.16853932584269665 and parameters: {'threshold': 0.43005905572277003}. Best is trial 0 with value: -0.16853932584269665.
[I 2025-12-14 20:39:47,032] Trial 1 finished with value: -0.16759776536312848 and parameters: {'threshold': 0.3591162556526585}. Best is trial 0 with value: -0.16853932584269665.
[I 2025-12-14 20:39:47,033] Trial 2 finished with value: -0.17751479289940827 and parameters: {'threshold': 0.5645288048513855}. Best is trial 2 with value: -0.17751479289940827.
[I 2025-12-14 20:39:47,033] Trial 3 finished with value: -0.16853932584269665 and parameters: {'threshold': 0.41167428108499104}. Best is trial 2 with value: -0.17751479289940827.
[I 2025-12-14 20:39:47,034] Trial 4 finished with value: -0.14285714285714285 and parameters: {'threshold': 0.919880485696478}. Best is trial 2 with value: -0.177514792

0.7949627149628297 0.37878787878787884


dspy framework เอาไว้หาว่า sentence ไหนดีที่สุด

# test

สิ่งสำคัญคือ trainset ต้องดี

ทำ keyword มาก่อนแล้วให้ ai gen คำจาก kw มา

ลองหา centroid ของ intent แล้วดูระยะห่างของคำ

In [ ]:
NEXT_ACTION_KEYWORDS = [
    "ทำอะไรต่อ", "แนะนำ", "เอาไปใช้", "ตัดสินใจ",
    "next", "action", "step ต่อไป", "ควรทำยังไง"
]

ANALYSIS_KEYWORDS = [
    "วิเคราะห์", "สรุป", "ดูแนวโน้ม", "เป็นเพราะอะไร",
    "เกิดจาก", "insight", "performance",
    "คืออะไร", "อธิบาย", "ต่างกันยังไง", "ทำงานยังไง"
]

def rule_intent(text):
    if any(k in text for k in NEXT_ACTION_KEYWORDS):
        return "next_action", 0.9
    if any(k in text for k in ANALYSIS_KEYWORDS):
        return "analysis", 0.9
    return None, 0.0


In [ ]:
text = "จากข้อมูลนี้ควรเอาไปใช้ยังไงต่อ"
rule_intent(text)

('next_action', 0.9)

https://chatgpt.com/share/693e5c8b-1050-8003-9c87-531f54dc77e0